# TARGE — Five-Group Ablation Sweep (Colab)

Runs the five ablation groups against a single **already-trained** TARGE checkpoint.
Pipeline: setup → install → login → paths → data → **split train/heldout** → checkpoint → oracle → sweep → table.

| Group | route_mode      | use_qformer | Notes |
|-------|-----------------|-------------|-------|
| A     | full            | False       | upper bound: all N tokens, no compression |
| B     | random_topk     | False       | lower bound: random k tokens |
| C     | oracle          | False       | indices precomputed from LLM cross-attention |
| D     | selector        | True        | trained behavior (status quo) |
| E     | selector        | False       | selector-only |

**Held-out set:** 500 image-text pairs split off from the LLaVA-CC-SBU training shard *before* training (seeded). The selector / projector never sees these during training, so the cos / IoU / generation pass is leak-free. POPE was removed — the metric set is now `cos_vs_A`, `iou_vs_oracle`, and qualitative generations.


## 1. Setup: GPU check + mount Drive


In [ ]:
print("== GPU ==")
!nvidia-smi -L || echo "(no GPU detected)"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true

print("\n== Drive ==")
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


## 2. Filepaths + install

Single source of truth for every path used in this notebook.

- **Drive** = `/content/drive/MyDrive/...` — persistent, survives session teardown, but slow (FUSE-backed). Use only for the authoritative source code (read) and durable outputs we want to keep (write).
- **Local** = `/content/...` — the VM's NVMe — fast, lost when the session ends. Everything compute-related (source mirror, dataset, HF cache, intermediates) lives here.

This cell also rsync-mirrors the source from Drive to local SSD and runs `pip install -e .` from the mirror, because `pip install -e` writes a lot of small files (egg-info, .pth) and that's painful on Drive.


In [ ]:
import os, time
from pathlib import Path

# ─── H100 / Ampere+ perf knobs (free speedup; safe everywhere) ──────────────
try:
    import torch
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True
    torch.set_float32_matmul_precision("high")
    if torch.cuda.is_available():
        print(f"[gpu] {torch.cuda.get_device_name(0)}  (TF32 on, cudnn.benchmark on)")
except Exception as _e:
    print(f"[gpu] perf knobs skipped: {_e}")


# ─── Tunable knobs ────────────────────────────────────────────────────────────
RUN_ID    = "targe-smollm2-75"
N_HELDOUT = 500
SPLIT_SEED = 7

# ─── Drive (persistent — source of truth + durable outputs) ──────────────────
DRIVE_ROOT   = Path("/content/drive/MyDrive/targe-prismatic-vlms")
CKPT_DIR     = DRIVE_ROOT / "runs" / RUN_ID
RESULTS_JSON = CKPT_DIR / "ablation_results.json"

# ─── Local (VM SSD — compute scratch) ────────────────────────────────────────
LOCAL_ROOT   = Path("/content")
REPO_DIR     = LOCAL_ROOT / "repo"
TAR_PATH     = LOCAL_ROOT / "train_subset_75.tar"
DATA_DIR     = LOCAL_ROOT / "data" / "download" / "llava-laion-cc-sbu-558k"
CHAT_JSON    = DATA_DIR / "chat.json"           # training subset (after split)
CHAT_FULL    = DATA_DIR / "chat_full.json"      # original pre-split (split-cell sentinel)
HELDOUT_JSON = DATA_DIR / "chat_heldout.json"   # 500 held-out for ablation eval
ORACLE_PT    = LOCAL_ROOT / "oracle_indices.pt"
HF_CACHE     = LOCAL_ROOT / "hf_cache"

os.chdir(LOCAL_ROOT)

for d in (DATA_DIR.parent, HF_CACHE, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"]      = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")

# ─── Mirror source Drive → local SSD (incremental rsync) ───────────────────
t0 = time.time()
REPO_DIR.mkdir(parents=True, exist_ok=True)
!rsync -a \
    --exclude='runs' \
    --exclude='data' \
    --exclude='hf_cache' \
    --exclude='*.tar' \
    --exclude='__pycache__' \
    --exclude='*.pyc' \
    --exclude='.git' \
    --exclude='*.egg-info' \
    --exclude='build' \
    --exclude='dist' \
    --exclude='.ipynb_checkpoints' \
    "{DRIVE_ROOT}/" "{REPO_DIR}/"
print(f"[mirror] {time.time()-t0:.1f}s")

PYPROJECT = REPO_DIR / "pyproject.toml"
assert PYPROJECT.exists(), (
    f"rsync finished but {PYPROJECT} is missing — likely a Drive FUSE crash mid-copy.\n"
    "Fix: re-mount Drive ( drive.mount('/content/drive', force_remount=True) ),\n"
    "     then `!rm -rf /content/repo` and re-run this cell."
)

%cd {REPO_DIR}
INSTALLED_MARKER = REPO_DIR / ".pip_installed"
if not INSTALLED_MARKER.exists():
    t0 = time.time()
    !pip install -e . --no-build-isolation --quiet
    !pip install -q fvcore || echo "[install] fvcore failed — FLOPS column will be null"
    INSTALLED_MARKER.touch()
    print(f"[install] {time.time()-t0:.1f}s")
else:
    print("[install] cached  (remove /content/repo/.pip_installed to force reinstall)")

print("\n" + "─" * 60)
print(f"Drive source : {DRIVE_ROOT}")
print(f"Drive ckpt   : {CKPT_DIR}  ({'exists' if CKPT_DIR.exists() else 'MISSING'})")
print(f"Drive results: {RESULTS_JSON}")
print(f"Local repo   : {REPO_DIR}")
print(f"Local tar    : {TAR_PATH}   ({'exists' if TAR_PATH.exists() else 'MISSING'})")
print(f"Local data   : {DATA_DIR}   ({'extracted' if DATA_DIR.exists() else 'not yet'})")
print(f"Heldout      : {HELDOUT_JSON}  ({'split done' if HELDOUT_JSON.exists() else 'not yet'})")
print(f"HF cache     : {HF_CACHE}")

_latest_ckpt = CKPT_DIR / "checkpoints" / "latest-checkpoint.pt"
assert CKPT_DIR.exists(), (
    f"CKPT_DIR `{CKPT_DIR}` does not exist on Drive. "
    f"Check RUN_ID (currently `{RUN_ID}`)."
)


## 3. Logins (Hugging Face, W&B)


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login, whoami

# Hugging Face
hf_token = userdata.get('hf_token')
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
who = whoami()
print(f"[hf] logged in as: {who.get('name')}  (orgs: {[o['name'] for o in who.get('orgs', [])]})")


## 4. Untar dataset into the layout `llava-v15` expects

Idempotent: skips if `chat.json` is already present. Extracts to `/content/data/download/llava-laion-cc-sbu-558k/` and renames `chat_subset_5k.json` → `chat.json` so the default `LLaVa_V15_Config` paths resolve.


In [ ]:
import time, tarfile

OLD_DIR = Path("/content/train_subset_75")  # legacy unpacked fallback

assert TAR_PATH.exists() or OLD_DIR.exists() or CHAT_JSON.exists(), (
    f"Need either {TAR_PATH}, {OLD_DIR}, or extracted {CHAT_JSON}."
)

if CHAT_JSON.exists():
    n_files = sum(1 for _ in DATA_DIR.rglob("*") if _.is_file())
    print(f"[skip] {DATA_DIR} already populated ({n_files:,} files)")
else:
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

    if OLD_DIR.exists() and any(OLD_DIR.iterdir()):
        print(f"[move] {OLD_DIR} -> {DATA_DIR}")
        if DATA_DIR.exists():
            DATA_DIR.rmdir()
        OLD_DIR.rename(DATA_DIR)
    else:
        # Peek at the tar to decide whether to strip a wrapper directory.
        with tarfile.open(TAR_PATH) as tf:
            tops = set()
            for i, m in enumerate(tf):
                tops.add(m.name.split("/", 1)[0])
                if i > 20:
                    break
        has_wrapper = len(tops) == 1 and next(iter(tops)) not in {"", "."}
        strip_flag = "--strip-components 1" if has_wrapper else ""
        print(f"[tar] top-level sample : {sorted(tops)[:5]}{'...' if len(tops) > 5 else ''}")
        print(f"[tar] strip wrapper dir? {has_wrapper}")

        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[tar] extracting -> {DATA_DIR}  ({TAR_PATH.stat().st_size / 1e9:.2f} GB)")
        t0 = time.time()
        !tar -xf "{TAR_PATH}" {strip_flag} -C "{DATA_DIR}"
        print(f"[tar] done in {time.time()-t0:.1f}s")

# The new tarball already ships chat.json under the expected name — no rename needed.
if not CHAT_JSON.exists():
    hits = list(Path("/content").rglob("chat.json"))
    raise FileNotFoundError(f"Expected {CHAT_JSON} but missing. Found candidates: {hits}")

print(f"\n[ok] {CHAT_JSON} ({CHAT_JSON.stat().st_size / 1e6:.2f} MB)")


## 4b. Split 500 image-text pairs off as held-out test set

Pulls `N_HELDOUT` examples out of `chat.json` (seeded shuffle), writes them to `chat_heldout.json`, and replaces `chat.json` with the training-only subset. The original is preserved as `chat_full.json` — that file's existence is the split sentinel (delete it to force a re-split with a different seed).

This runs **before** training so the held-out images are never seen by the selector / projector.

In [ ]:
import json, random

if CHAT_FULL.exists() and HELDOUT_JSON.exists():
    with open(CHAT_JSON) as f: n_train = len(json.load(f))
    with open(HELDOUT_JSON) as f: n_held = len(json.load(f))
    print(f"[skip] split already done — train={n_train:,}  heldout={n_held:,}")
else:
    assert CHAT_JSON.exists(), f"Expected {CHAT_JSON} — run the untar cell first."
    with open(CHAT_JSON) as f:
        entries = json.load(f)
    assert len(entries) > N_HELDOUT, f"chat.json only has {len(entries):,} rows; need > {N_HELDOUT}"

    rng = random.Random(SPLIT_SEED)
    idx = list(range(len(entries)))
    rng.shuffle(idx)
    held_idx  = set(idx[:N_HELDOUT])
    held      = [entries[i] for i in idx[:N_HELDOUT]]
    train     = [entries[i] for i in range(len(entries)) if i not in held_idx]

    # Preserve the original under chat_full.json (sentinel), then overwrite chat.json
    # with the training subset so prismatic's default loader picks it up.
    CHAT_JSON.rename(CHAT_FULL)
    with open(CHAT_JSON, "w") as f:    json.dump(train, f)
    with open(HELDOUT_JSON, "w") as f: json.dump(held,  f)

    print(f"[split] seed={SPLIT_SEED}")
    print(f"[split] full     : {CHAT_FULL}    ({len(entries):,} rows, preserved)")
    print(f"[split] train    : {CHAT_JSON}    ({len(train):,} rows)")
    print(f"[split] heldout  : {HELDOUT_JSON}  ({len(held):,} rows)")


## 4c. Pre-train sanity check: SmolLM2 chat template + label alignment

Runs `scripts/check_smollm2_labels.py` to confirm:
- `SmolLM2ChatPromptBuilder` produces real ChatML (`<|im_start|>` / `<|im_end|>`).
- The position-0 token that Prismatic force-ignores is a control token (not a content token we want to train on).
- User-turn tokens are masked to -100; assistant-turn tokens (including trailing `<|im_end|>`) are supervised.

If anything prints `WARN` / `FAIL`, stop and fix before training.

In [ ]:
%cd {REPO_DIR}
!python scripts/check_smollm2_labels.py

## 5. Train the connector (align stage)

In [ ]:
import os, time
os.environ["PYTHONUNBUFFERED"] = "1"

# ─── Training hyperparameters (align stage) ──────────────────────────────────
MODEL_TYPE    = "targe-smollm2-360m-clipb-224px"
STAGE         = "align"
BSZ           = 14                  # per-device == global (single GPU)
LR            = 1e-4
MAX_STEPS     = 500
DATASET_TYPE  = "llava-v15"
WANDB_ENTITY  = "nbusich-northeastern-university"
WANDB_PROJECT = "targe"

LATEST_CKPT = CKPT_DIR / "checkpoints" / "latest-checkpoint.pt"
TRAIN_LOG   = CKPT_DIR / "train.log"

if LATEST_CKPT.exists():
    print(f"[skip] checkpoint already exists: {LATEST_CKPT}")
    print(f"       (delete the file to force retrain)")
else:
    HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
    WB_KEY_VAL   = os.environ.get("WANDB_API_KEY", "").strip()
    TRACKERS     = "[jsonl,wandb]" if WB_KEY_VAL else "[jsonl]"
    TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)

    PRETRAIN = REPO_DIR / "scripts" / "pretrain.py"

    print(f"[train] run_id    : {RUN_ID}")
    print(f"[train] model     : {MODEL_TYPE}")
    print(f"[train] stage     : {STAGE}")
    print(f"[train] bsz/lr    : {BSZ} / {LR}")
    print(f"[train] max_steps : {MAX_STEPS}")
    print(f"[train] data      : {DATA_DIR}")
    print(f"[train] ckpts to  : {CKPT_DIR}/checkpoints/  (Drive — durable)")
    print(f"[train] log to    : {TRAIN_LOG}")
    print(f"[train] wandb     : {'on (' + WANDB_PROJECT + ')' if WB_KEY_VAL else 'off (no WANDB_API_KEY)'}")
    print(f"[train] started   : {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

    # Launch from /content so the LLaVa_V15_Config default `data/…` resolves to {DATA_DIR}.
    %cd {LOCAL_ROOT}
    !set -o pipefail; HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" WANDB_API_KEY="{WB_KEY_VAL}" HF_HOME="{HF_CACHE}" \
      stdbuf -oL -eL torchrun --standalone --nnodes 1 --nproc-per-node 1 "{PRETRAIN}" \
        --model.type "{MODEL_TYPE}" \
        --stage "{STAGE}" \
        --model.align_per_device_batch_size {BSZ} \
        --model.align_global_batch_size {BSZ} \
        --model.align_learning_rate {LR} \
        --model.align_max_steps {MAX_STEPS} \
        --dataset.type "{DATASET_TYPE}" \
        --run_id "{RUN_ID}" \
        --run_root_dir "{CKPT_DIR.parent}" \
        --hf_token HF_TOKEN \
        --trackers '{TRACKERS}' \
        --wandb_entity "{WANDB_ENTITY}" \
        --wandb_project "{WANDB_PROJECT}" 2>&1 | tee -a "{TRAIN_LOG}"

    print(f"\n[train] finished  : {time.strftime('%Y-%m-%d %H:%M:%S')}")
    assert LATEST_CKPT.exists(), (
        f"Training process exited but {LATEST_CKPT} is missing — check {TRAIN_LOG} for errors."
    )


## 5 (MLP baseline). Train the same model with a plain MLP connector

Apples-to-apples baseline against section 5: identical config, same hyperparameters, same data — but the connector is a vanilla `MLPProjector` (`Linear → GELU → Linear`) instead of the selector + Q-Former pipeline. Achieved by overriding `--model.arch_specifier` to `no-align+gelu-mlp` on the same `targe-smollm2-360m-clipb-224px` base config. Writes to its own `RUN_ID_MLP` directory on Drive so it does not collide with the selector checkpoint.

In [ ]:
import os, time
os.environ["PYTHONUNBUFFERED"] = "1"

# ─── Training hyperparameters (align stage) — match section 5 exactly ────────
RUN_ID_MLP    = "targe-smollm2-75-mlp"
MODEL_TYPE    = "targe-smollm2-360m-clipb-224px"
ARCH_MLP      = "no-align+gelu-mlp"     # ← swaps SelectorCompressorPipeline → MLPProjector
STAGE         = "align"
BSZ           = 14                  # per-device == global (single GPU)
LR            = 1e-4
MAX_STEPS     = 500
DATASET_TYPE  = "llava-v15"
WANDB_ENTITY  = "nbusich-northeastern-university"
WANDB_PROJECT = "targe"

CKPT_DIR_MLP    = DRIVE_ROOT / "runs" / RUN_ID_MLP
LATEST_CKPT_MLP = CKPT_DIR_MLP / "checkpoints" / "latest-checkpoint.pt"
TRAIN_LOG_MLP   = CKPT_DIR_MLP / "train.log"
CKPT_DIR_MLP.mkdir(parents=True, exist_ok=True)

if LATEST_CKPT_MLP.exists():
    print(f"[skip] checkpoint already exists: {LATEST_CKPT_MLP}")
    print(f"       (delete the file to force retrain)")
else:
    HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
    WB_KEY_VAL   = os.environ.get("WANDB_API_KEY", "").strip()
    TRACKERS     = "[jsonl,wandb]" if WB_KEY_VAL else "[jsonl]"
    TRAIN_LOG_MLP.parent.mkdir(parents=True, exist_ok=True)

    PRETRAIN = REPO_DIR / "scripts" / "pretrain.py"

    print(f"[train] run_id    : {RUN_ID_MLP}")
    print(f"[train] model     : {MODEL_TYPE}  (arch override: {ARCH_MLP})")
    print(f"[train] stage     : {STAGE}")
    print(f"[train] bsz/lr    : {BSZ} / {LR}")
    print(f"[train] max_steps : {MAX_STEPS}")
    print(f"[train] data      : {DATA_DIR}")
    print(f"[train] ckpts to  : {CKPT_DIR_MLP}/checkpoints/  (Drive — durable)")
    print(f"[train] log to    : {TRAIN_LOG_MLP}")
    print(f"[train] wandb     : {'on (' + WANDB_PROJECT + ')' if WB_KEY_VAL else 'off (no WANDB_API_KEY)'}")
    print(f"[train] started   : {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

    # Launch from /content so the LLaVa_V15_Config default `data/…` resolves to {DATA_DIR}.
    %cd {LOCAL_ROOT}
    !set -o pipefail; HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" WANDB_API_KEY="{WB_KEY_VAL}" HF_HOME="{HF_CACHE}" \
      stdbuf -oL -eL torchrun --standalone --nnodes 1 --nproc-per-node 1 "{PRETRAIN}" \
        --model.type "{MODEL_TYPE}" \
        --model.arch_specifier "{ARCH_MLP}" \
        --stage "{STAGE}" \
        --model.align_per_device_batch_size {BSZ} \
        --model.align_global_batch_size {BSZ} \
        --model.align_learning_rate {LR} \
        --model.align_max_steps {MAX_STEPS} \
        --dataset.type "{DATASET_TYPE}" \
        --run_id "{RUN_ID_MLP}" \
        --run_root_dir "{CKPT_DIR_MLP.parent}" \
        --hf_token HF_TOKEN \
        --trackers '{TRACKERS}' \
        --wandb_entity "{WANDB_ENTITY}" \
        --wandb_project "{WANDB_PROJECT}" 2>&1 | tee -a "{TRAIN_LOG_MLP}"

    print(f"\n[train] finished  : {time.strftime('%Y-%m-%d %H:%M:%S')}")
    assert LATEST_CKPT_MLP.exists(), (
        f"Training process exited but {LATEST_CKPT_MLP} is missing — check {TRAIN_LOG_MLP} for errors."
    )


## 5b. Locate the trained checkpoint\n\nLists all checkpoints under `CKPT_DIR` (on Drive). The ablation scripts load the model via `prismatic.load(CKPT_DIR)`, which picks up `config.json` + the latest checkpoint under `checkpoints/`.

In [ ]:
ckpts = sorted((CKPT_DIR / "checkpoints").glob("*.pt"))
print(f"[ckpt] run dir   : {CKPT_DIR}")
print(f"[ckpt] checkpoints found ({len(ckpts)}):")
for c in ckpts:
    sz = c.stat().st_size / 1e9
    print(f"   - {c.name:40s}  {sz:6.2f} GB")

assert ckpts, f"No checkpoints found in {CKPT_DIR / 'checkpoints'}. Train first (TARGE.ipynb section 7) or fix RUN_ID."
LATEST = ckpts[-1]
print(f"\n[ckpt] latest    : {LATEST.name}")


## 6. Precompute oracle indices

For each held-out example, run a forward with `route_mode="full"` + `use_qformer=False` so the LLM sees every projected visual token. Then average attention from response-token positions to visual-token positions across the early layers and take top-k. Saved as `ORACLE_PT` (local — intermediate, not persisted to Drive; cheap to recompute).

In [ ]:
# Drop any stale oracle file — IDs depend on the heldout source, so a session reset is safest.
ORACLE_PT.unlink(missing_ok=True)

%cd {REPO_DIR}
HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
!HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" HF_HOME="{HF_CACHE}" \
  python scripts/eval/precompute_oracle.py \
    --model_path "{CKPT_DIR}" \
    --heldout_json "{HELDOUT_JSON}" \
    --image_root "{DATA_DIR}" \
    --out_pt "{ORACLE_PT}" \
    --hf_token HF_TOKEN


## 7. Run the 5-group ablation sweep

For each held-out example, runs all five routing configurations against the same checkpoint, capturing generations, connector-output latents (for cosine similarity vs. Group A), Oracle IoU (groups D, E), and per-group hardware costs (FLOPS via `fvcore` if available, latency via CUDA events). Results are flushed atomically to `RESULTS_JSON` (on Drive — durable) after every example.

POPE accuracy is no longer reported — the held-out set is now training-distribution captions, not yes/no QA.

In [ ]:
EVAL_BATCH_SIZE = 64
LOG_EVERY       = 50
ABLATION_LOG    = CKPT_DIR / "ablation_eval.log"

%cd {REPO_DIR}
HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
!HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" HF_HOME="{HF_CACHE}" \
  python scripts/eval/run_ablation.py \
    --model_path "{CKPT_DIR}" \
    --heldout_json "{HELDOUT_JSON}" \
    --image_root "{DATA_DIR}" \
    --oracle_pt "{ORACLE_PT}" \
    --out_json "{RESULTS_JSON}" \
    --max_new_tokens 16 \
    --eval_batch_size {EVAL_BATCH_SIZE} \
    --log_every {LOG_EVERY} \
    --hf_token HF_TOKEN 2>&1 | tee -a "{ABLATION_LOG}"


## 8. Results table

In [ ]:
import json
import pandas as pd

with open(RESULTS_JSON) as f:
    res = json.load(f)

rows = []
for name, s in res["summary"].items():
    hw = s.get("hardware") or {}
    rows.append({
        "group":             name,
        "n":                 s.get("n_generations"),
        "cos_vs_A":          s.get("cos_vs_A_mean"),
        "iou_vs_oracle":     s.get("iou_vs_oracle_mean"),
        "flops":             hw.get("flops"),
        "latency_ms_median": hw.get("latency_ms_median"),
    })
df = pd.DataFrame(rows).set_index("group")
print(df.to_string(float_format=lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)))


### 8b. Spot-check generations\n\nCompare what each group says for a few held-out prompts.

In [ ]:
from IPython.display import display, Markdown

gens = res["generations"]
group_names = list(gens.keys())
# Index by example id within each group so we can align across groups.
by_id = {g: {item["id"]: item for item in gens[g]} for g in group_names}
common_ids = sorted(set.intersection(*[set(d.keys()) for d in by_id.values()]))
print(f"[gens] {len(common_ids)} examples present in every group")

for ex_id in common_ids[:3]:
    prompt = by_id[group_names[0]][ex_id]["prompt"]
    lines = [f"**id:** `{ex_id}`", f"**prompt:** {prompt}", ""]
    for g in group_names:
        lines.append(f"- **{g}:** {by_id[g][ex_id]['gen']}")
    display(Markdown("\n".join(lines)))
    display(Markdown("---"))
